# Prompting for clinical tasks

Exercise notebook for the lesson **[Prompting for clinical tasks](https://ramkumarmd.com/medical-ai/generative-ai-llms/prompting-for-clinical-tasks)** in the *Generative AI & LLMs* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** A good clinical prompt states the role and task, gives the relevant context, sets explicit limits (use only the text provided, say 'not stated' rather than guess), specifies an exact output format and includes a worked example. Prompting is not a one-off craft: treat each prompt as a version to be tested on labelled synthetic cases, and validate every output in code before it is used.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Building a few-shot prompt

A system message, one worked example and the new synthetic note, as chat messages.

In [ ]:
SYSTEM = """You extract information from clinic notes for a diabetes registry.
Return only JSON with exactly these keys:
  "diabetes_type": one of "type 1", "type 2", "gestational", "other", "not stated"
  "latest_hba1c_percent": a number, or null if not stated
  "on_insulin": true, false, or null if not stated
Use only information written in the note. Do not guess."""

EXAMPLE_NOTE = "54F, T2DM for 8 yrs on metformin. HbA1c today 8.1%. No insulin."
EXAMPLE_ANSWER = '{"diabetes_type": "type 2", "latest_hba1c_percent": 8.1, "on_insulin": false}'

def build_messages(note):
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Note:\n{EXAMPLE_NOTE}"},     # one worked example
        {"role": "assistant", "content": EXAMPLE_ANSWER},
        {"role": "user", "content": f"Note:\n{note}"},
    ]

messages = build_messages("Synthetic note: 31M, type 1 since age 12, basal-bolus insulin. A1c 7.4% (Mar).")
for m in messages:
    print(f"[{m['role']}] " + m["content"].replace("\n", " ")[:70])

# [system] You extract information from clinic notes for a diabetes registry. Ret
# [user] Note: 54F, T2DM for 8 yrs on metformin. HbA1c today 8.1%. No insulin.
# [assistant] {"diabetes_type": "type 2", "latest_hba1c_percent": 8.1, "on_insulin":
# [user] Note: Synthetic note: 31M, type 1 since age 12, basal-bolus insulin. A

### Validating every output

Schema and plausibility checks that reject malformed or implausible outputs.

In [ ]:
import json

ALLOWED_TYPES = {"type 1", "type 2", "gestational", "other", "not stated"}

def validate(raw):
    """Return (record, problems). Never trust model output without checking it."""
    try:
        record = json.loads(raw)
    except json.JSONDecodeError:
        return None, ["not valid JSON"]
    problems = []
    if set(record) != {"diabetes_type", "latest_hba1c_percent", "on_insulin"}:
        problems.append(f"unexpected keys: {sorted(record)}")
    if record.get("diabetes_type") not in ALLOWED_TYPES:
        problems.append(f"diabetes_type not allowed: {record.get('diabetes_type')!r}")
    a1c = record.get("latest_hba1c_percent")
    if a1c is not None and not (isinstance(a1c, (int, float)) and 3 <= a1c <= 20):
        problems.append(f"implausible HbA1c: {a1c!r}")
    if record.get("on_insulin") not in (True, False, None):
        problems.append("on_insulin must be true, false or null")
    return record, problems

outputs = [
    '{"diabetes_type": "type 1", "latest_hba1c_percent": 7.4, "on_insulin": true}',
    'Sure! Here is the JSON: {"diabetes_type": "type 2"}',
    '{"diabetes_type": "Type II", "latest_hba1c_percent": 74, "on_insulin": "yes"}',
]
for raw in outputs:
    record, problems = validate(raw)
    print("OK" if not problems else "REJECT", problems)

# OK []
# REJECT ['not valid JSON']
# REJECT ["diabetes_type not allowed: 'Type II'", 'implausible HbA1c: 74', 'on_insulin must be true, false or null']

### Testing prompts on labelled cases

Field-level accuracy for two prompt versions on the same four labelled synthetic notes.

In [ ]:
gold = [
    {"diabetes_type": "type 1", "latest_hba1c_percent": 7.4, "on_insulin": True},
    {"diabetes_type": "type 2", "latest_hba1c_percent": None, "on_insulin": False},
    {"diabetes_type": "gestational", "latest_hba1c_percent": 5.6, "on_insulin": None},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 9.2, "on_insulin": True},
]
prompt_v1 = [   # validated outputs from the first prompt version
    {"diabetes_type": "type 1", "latest_hba1c_percent": 7.4, "on_insulin": True},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 7.0, "on_insulin": False},  # invented value
    {"diabetes_type": "type 2", "latest_hba1c_percent": 5.6, "on_insulin": False},  # wrong type, guessed
    {"diabetes_type": "type 2", "latest_hba1c_percent": 9.2, "on_insulin": True},
]
prompt_v2 = [   # after adding "Do not guess" and the null rules
    {"diabetes_type": "type 1", "latest_hba1c_percent": 7.4, "on_insulin": True},
    {"diabetes_type": "type 2", "latest_hba1c_percent": None, "on_insulin": False},
    {"diabetes_type": "gestational", "latest_hba1c_percent": 5.6, "on_insulin": None},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 9.2, "on_insulin": False},  # missed insulin
]

def field_accuracy(predictions):
    fields = gold[0].keys()
    return {f: sum(p[f] == g[f] for p, g in zip(predictions, gold)) / len(gold) for f in fields}

for name, preds in [("prompt v1", prompt_v1), ("prompt v2", prompt_v2)]:
    print(name, field_accuracy(preds))

# prompt v1 {'diabetes_type': 0.75, 'latest_hba1c_percent': 0.75, 'on_insulin': 0.75}
# prompt v2 {'diabetes_type': 1.0, 'latest_hba1c_percent': 1.0, 'on_insulin': 0.75}

## Exercise data

Synthetic data written for this exercise (no real patients). Run this cell before starting the steps.

In [ ]:
# Twenty synthetic clinic notes (no real patients), with clinician-style labels for the three fields.
# Labelling rules: "on_insulin" is true if the patient is currently using insulin, false only if the note
# says they are not, otherwise null; "latest_hba1c_percent" is the most recent value in %, or null.
notes = [
    "62M T2DM x12y on metformin 1 g BD + sitagliptin. HbA1c 7.8% (Jan). No insulin.",
    "28F type 1 since childhood, insulin pump. A1c 8.2%.",
    "34F G2P1 at 28 weeks, GDM on diet alone. HbA1c 5.4%.",
    "71F long-standing type 2 diabetes. Insulin glargine 18 units at night. HbA1c pending.",
    "45M seen for hypertension review. Diabetes noted in history. No recent HbA1c.",
    "39M LADA, GAD antibodies positive, started basal insulin last month. HbA1c 9.1%.",
    "55F T2DM. HbA1c 6.8% in March, 7.3% today.",
    "67M T2DM, insulin stopped in 2024 after weight loss; now on metformin only. A1c 6.5%.",
    "22F T1DM, MDI (glargine + aspart). HbA1c 10.4%, missing doses.",
    "50M MODY (HNF1A) on gliclazide. HbA1c 7.0%.",
    "31F, 34/40, gestational diabetes, now on isophane insulin 8 u at night. Last HbA1c 5.9%.",
    "58M type 2, declined insulin today; plan GLP-1 receptor agonist. HbA1c 9.6%.",
    "76F T2DM. Frail. HbA1c 6.1%. Gliclazide reduced. Not on insulin.",
    "40M diabetes, type unclear: awaiting C-peptide and antibodies. HbA1c 8.8%. On metformin.",
    "63F T2DM 15 yrs, premixed insulin 30/70 twice daily. HbA1c 8.4%.",
    "19M T1DM, hybrid closed-loop system. HbA1c 6.9%.",
    "48F prediabetes, HbA1c 6.1%. Lifestyle advice given.",
    "69M T2DM. HbA1c 7.5% three months ago; not repeated today. Uses insulin degludec.",
    "36F diabetes after chronic pancreatitis (type 3c), on insulin. A1c 8.0%.",
    "57M T2DM. Previous HbA1c 8.9%, today 8.1%. On empagliflozin + metformin; no insulin.",
]

labels = [
    {"diabetes_type": "type 2", "latest_hba1c_percent": 7.8, "on_insulin": False},
    {"diabetes_type": "type 1", "latest_hba1c_percent": 8.2, "on_insulin": True},
    {"diabetes_type": "gestational", "latest_hba1c_percent": 5.4, "on_insulin": False},
    {"diabetes_type": "type 2", "latest_hba1c_percent": None, "on_insulin": True},
    {"diabetes_type": "not stated", "latest_hba1c_percent": None, "on_insulin": None},
    {"diabetes_type": "other", "latest_hba1c_percent": 9.1, "on_insulin": True},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 7.3, "on_insulin": None},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 6.5, "on_insulin": False},
    {"diabetes_type": "type 1", "latest_hba1c_percent": 10.4, "on_insulin": True},
    {"diabetes_type": "other", "latest_hba1c_percent": 7.0, "on_insulin": None},
    {"diabetes_type": "gestational", "latest_hba1c_percent": 5.9, "on_insulin": True},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 9.6, "on_insulin": False},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 6.1, "on_insulin": False},
    {"diabetes_type": "not stated", "latest_hba1c_percent": 8.8, "on_insulin": None},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 8.4, "on_insulin": True},
    {"diabetes_type": "type 1", "latest_hba1c_percent": 6.9, "on_insulin": True},
    {"diabetes_type": "not stated", "latest_hba1c_percent": 6.1, "on_insulin": None},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 7.5, "on_insulin": True},
    {"diabetes_type": "other", "latest_hba1c_percent": 8.0, "on_insulin": True},
    {"diabetes_type": "type 2", "latest_hba1c_percent": 8.1, "on_insulin": False},
]

# "Recorded outputs" for readers without an approved model. These are SIMULATED, written to show errors
# that language models commonly make on this kind of task; they are not outputs from any real model.
# v1: the lesson's prompt with one example. v2: after tightening the rules (see the solution notebook).
recorded_v1 = [
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 7.8, "on_insulin": false}',
    '{"diabetes_type": "type 1", "latest_hba1c_percent": 8.2, "on_insulin": true}',
    '{"diabetes_type": "gestational", "latest_hba1c_percent": 5.4, "on_insulin": null}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": null, "on_insulin": true}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": null, "on_insulin": false}',
    '{"diabetes_type": "type 1", "latest_hba1c_percent": 9.1, "on_insulin": true}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 6.8, "on_insulin": false}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 6.5, "on_insulin": true}',
    'Here is the extracted data:\n{"diabetes_type": "type 1", "latest_hba1c_percent": 10.4, "on_insulin": true}',
    '{"diabetes_type": "MODY", "latest_hba1c_percent": 7.0, "on_insulin": false}',
    '{"diabetes_type": "gestational", "latest_hba1c_percent": 5.9, "on_insulin": true}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 9.6, "on_insulin": true}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 6.1, "on_insulin": false}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 8.8, "on_insulin": false}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 8.4, "on_insulin": true}',
    '{"diabetes_type": "type 1", "latest_hba1c_percent": 6.9, "on_insulin": true}',
    '{"diabetes_type": "other", "latest_hba1c_percent": 6.1, "on_insulin": false}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": "7.5%", "on_insulin": true}',
    '{"diabetes_type": "other", "latest_hba1c_percent": 8.0, "on_insulin": true}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 8.1, "on_insulin": false}',
]
recorded_v2 = [
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 7.8, "on_insulin": false}',
    '{"diabetes_type": "type 1", "latest_hba1c_percent": 8.2, "on_insulin": true}',
    '{"diabetes_type": "gestational", "latest_hba1c_percent": 5.4, "on_insulin": null}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": null, "on_insulin": true}',
    '{"diabetes_type": "not stated", "latest_hba1c_percent": null, "on_insulin": null}',
    '{"diabetes_type": "other", "latest_hba1c_percent": 9.1, "on_insulin": true}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 7.3, "on_insulin": null}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 6.5, "on_insulin": false}',
    '{"diabetes_type": "type 1", "latest_hba1c_percent": 10.4, "on_insulin": true}',
    '{"diabetes_type": "other", "latest_hba1c_percent": 7.0, "on_insulin": null}',
    '{"diabetes_type": "gestational", "latest_hba1c_percent": 5.9, "on_insulin": true}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 9.6, "on_insulin": false}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 6.1, "on_insulin": false}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 8.8, "on_insulin": null}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 8.4, "on_insulin": true}',
    '{"diabetes_type": "type 1", "latest_hba1c_percent": 6.9, "on_insulin": true}',
    '{"diabetes_type": "not stated", "latest_hba1c_percent": 6.1, "on_insulin": null}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 7.5, "on_insulin": true}',
    '{"diabetes_type": "other", "latest_hba1c_percent": 8.0, "on_insulin": true}',
    '{"diabetes_type": "type 2", "latest_hba1c_percent": 8.1, "on_insulin": false}',
]
print(f"{len(notes)} notes, {len(labels)} labels, {len(recorded_v1)} + {len(recorded_v2)} recorded outputs")

## Exercise

**Goal:** Design, validate and evaluate a prompt that extracts diabetes information from synthetic clinic notes.

**Data:** Twenty synthetic clinic notes with clinician-style labels, written for the notebook (Created for this course; no real patient data)

### Step 1

Write a system prompt and two few-shot examples for the three-field extraction task.

In [ ]:
# Your code here


### Step 2

Run the prompt through an approved model (or the recorded outputs provided in the notebook) on all 20 notes.

In [ ]:
# Your code here


### Step 3

Validate every output with the schema checks and count how many are rejected and why.

In [ ]:
# Your code here


### Step 4

Score field-level accuracy against the labels, then change one thing in the prompt and re-score.

In [ ]:
# Your code here


### Step 5

Add three adversarial notes (a negation, an mmol/mol HbA1c and an embedded instruction) and check how the prompt copes.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/generative-ai-llms/prompting-for-clinical-tasks](https://ramkumarmd.com/medical-ai/generative-ai-llms/prompting-for-clinical-tasks). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.